## Setup and Data Loading

In [2]:
import pandas as pd

# Load the datasets
intakes = pd.read_csv('aac_intakes.csv')
outcomes = pd.read_csv('aac_outcomes.csv')
combined = pd.read_csv('aac_intakes_outcomes.csv') # Useful for age-based analysis

# Convert datetime columns to actual datetime objects
intakes['datetime'] = pd.to_datetime(intakes['datetime'])
outcomes['datetime'] = pd.to_datetime(outcomes['datetime'])

## 1. Analysis of Locations. I grouped the data by the "Found Location" column, counted the occurrences, and sorted them in descending order to isolate the top 5.

In [4]:
# Question 1: Top 5 Found Locations
top_locations = intakes['found_location'].value_counts().head(5)

print("Top 5 Found Locations:")
print(top_locations)

Top 5 Found Locations:
found_location
Austin (TX)                          14443
Outside Jurisdiction                   948
Travis (TX)                            921
7201 Levander Loop in Austin (TX)      517
Del Valle (TX)                         411
Name: count, dtype: int64


## 2. Monthly Average and Trends (2015) I filtered the data for the year 2015, grouped the records by month, and calculated the mean. I also identified which specific months peaked in animal intake.

In [5]:
# Question 2: 2015 Monthly Intake Trends
intakes_2015 = intakes[intakes['datetime'].dt.year == 2015]
monthly_counts = intakes_2015.groupby(intakes_2015['datetime'].dt.month).size()

print(f"Average pets found per month in 2015: {monthly_counts.mean():.2f}")
print("\nMonthly totals (1=Jan, 12=Dec):")
print(monthly_counts.sort_values(ascending=False))

Average pets found per month in 2015: 1559.33

Monthly totals (1=Jan, 12=Dec):
datetime
6     2189
5     2094
10    1740
8     1718
7     1635
9     1591
4     1543
11    1411
3     1346
1     1198
12    1128
2     1119
dtype: int64


## 3. Incoming vs Adopted Ratio. I counted total entries (total rows) and compared them to rows where the "Outcome Type" was labeled as 'Adoption'.

In [6]:
# Question 3: Incoming vs. Adopted Ratio
total_in = len(intakes)
total_adopted = len(outcomes[outcomes['outcome_type'] == 'Adoption'])

ratio = total_adopted / total_in
print(f"Adoption Ratio: {ratio:.2f}")
print(f"Outcome: {ratio*100:.1f}% of incoming animals result in adoption.")

Adoption Ratio: 0.43
Outcome: 42.7% of incoming animals result in adoption.


## 4. Distribution of Animal Types. I performed a frequency count on the "Animal Type" column to see the total distribution.

In [7]:
# Question 4: Distribution of Animal Types
animal_dist = outcomes['animal_type'].value_counts()

print("Distribution of Animal Types:")
print(animal_dist)

Distribution of Animal Types:
animal_type
Dog          45856
Cat          30028
Other         4446
Bird           341
Livestock       10
Name: count, dtype: int64


## 5. Adoption Rates by Top 5 Breeds. I found the top 5 breeds by count, then calculated the percentage of "Adoption" outcomes specifically for those five breeds.

In [8]:
# Question 5: Adoption Rates for Dog Breeds
dogs = outcomes[outcomes['animal_type'] == 'Dog']
top_breeds = dogs['breed'].value_counts().head(5).index

for breed in top_breeds:
    breed_data = dogs[dogs['breed'] == breed]
    rate = (breed_data['outcome_type'] == 'Adoption').mean()
    print(f"{breed}: {rate:.1%} adoption rate")

Pit Bull Mix: 37.4% adoption rate
Chihuahua Shorthair Mix: 47.0% adoption rate
Labrador Retriever Mix: 49.6% adoption rate
German Shepherd Mix: 47.6% adoption rate
Australian Cattle Dog Mix: 55.7% adoption rate


## 6. Adoption Rates by Top 5 Colorings. Similar to the breed analysis, I identified the top 5 most common colors and calculated the adoption percentage for each.

In [9]:
# Question 6: Adoption Rates for Colors
top_colors = outcomes['color'].value_counts().head(5).index

for col in top_colors:
    color_data = outcomes[outcomes['color'] == col]
    rate = (color_data['outcome_type'] == 'Adoption').mean()
    print(f"{col}: {rate:.1%} adoption rate")

Black/White: 45.4% adoption rate
Black: 40.7% adoption rate
Brown Tabby: 42.2% adoption rate
Brown: 22.2% adoption rate
White: 37.8% adoption rate


## 7. Spay/Neuter Monthly Estimates. I filtered for animals listed as "Intact Male" or "Intact Female" and averaged these counts across the available months.

In [10]:
# Question 7: Monthly Spay/Neuter Volume
intact = outcomes[outcomes['sex_upon_outcome'].str.contains('Intact', na=False)]
monthly_surgery = intact.groupby(intact['datetime'].dt.to_period('M')).size()

print(f"Average spay/neuter procedures needed per month: {monthly_surgery.mean():.2f}")

Average spay/neuter procedures needed per month: 348.40


## Extra Credit: Repeats, Age Groups, and Costs. 
1. Used value_counts() on Animal IDs to find repeats.
2. Created a custom function to bin ages into Baby, Young, Adult, and Senior.
3. Applied fixed costs ($100 for dogs, $50 for cats) to all intact animals from 2015.

In [13]:
# Extra Credit: Repeats, Age Groups, and 2015 Surgery Costs

# 1. Repeats
# Count how many times each animal_id appears in the intakes
intake_counts = intakes['animal_id'].value_counts()
repeats = intake_counts[intake_counts > 1]

print(f"Number of repeat animals: {len(repeats)}")
print(f"Animal returned the most: {intake_counts.idxmax()} ({intake_counts.max()} times)")

# 2. Age Group Adoption Rates
outcomes['age_group'] = 'senior' # Default to senior
outcomes.loc[outcomes['age_upon_outcome'].str.contains('day|week|month', na=False), 'age_group'] = 'baby'
outcomes.loc[outcomes['age_upon_outcome'].str.contains('1 year|2 years', na=False), 'age_group'] = 'young'
outcomes.loc[outcomes['age_upon_outcome'].str.contains('3 years|4 years|5 years|6 years|7 years|8 years|9 years|10 years', na=False), 'age_group'] = 'adult'

age_adoption = outcomes.groupby('age_group')['outcome_type'].apply(lambda x: (x == 'Adoption').mean())

print("Adoption Rates by Age Group:")
print("-" * 30)
print(age_adoption.apply(lambda x: f"{x:.1%}"))

# 3. 2015 Surgery Costs
# 1. Filter intakes for the year 2015 and 'Intact' status
intact_2015 = intakes[(intakes['datetime'].dt.year == 2015) & 
                      (intakes['sex_upon_intake'].str.contains('Intact', na=False))]

# 2. Count the number of Dogs and Cats in that filtered list
dogs_count = len(intact_2015[intact_2015['animal_type'] == 'Dog'])
cats_count = len(intact_2015[intact_2015['animal_type'] == 'Cat'])

# 3. Calculate the costs (using the same names as above!)
total_cost = (dogs_count * 100) + (cats_count * 50)

# 4. Print the final answer
print(f"Number of Intact Dogs in 2015: {dogs_count}")
print(f"Number of Intact Cats in 2015: {cats_count}")
print(f"Total spent on Spay/Neuter in 2015: ${total_cost:,}")

Number of repeat animals: 6154
Animal returned the most: A721033 (13 times)
Adoption Rates by Age Group:
------------------------------
age_group
adult     32.7%
baby      51.7%
senior     3.4%
young     38.2%
Name: outcome_type, dtype: str
Number of Intact Dogs in 2015: 6100
Number of Intact Cats in 2015: 5065
Total spent on Spay/Neuter in 2015: $863,250
